# OBJECTIVES 

Incremental mean or other metric for the definition of heat wave

# RIVER INTRO

River is a Python library designed for online and incremental Machine Learning. Its API reflects the streaming mindset.

Typical operations include:

- `predict_one(x)` — predict one observation,
- `learn_one(x, y)` — learn from one observation,
- online metrics that update one result at a time,
- incremental preprocessing,
- drift detection,
- clustering,
- anomaly detection,
- ensembles.

The important pattern is: PREDICT → EVALUATE → LEARN → REPEAT

# COMPARE MENTAL MODELS OF BATCH AND STREAMING

#### Batch Machine Learning

```python
model.fit(X_train, y_train)
predictions = model.predict(X_test)
```

#### Streaming Machine Learning

```python
for x, y in stream:
    prediction = model.predict_one(x)
    metric.update(y, prediction)
    model.learn_one(x, y)
```

Do not worry about memorizing the River syntax today.

Focus on the sequence: **The model is evaluated and updated while the data is arriving.**

# REUSABLE PATTERNS

#### Pattern A — Process a stream

```python
for observation in stream:
    process(observation)
```

#### Pattern B — Update a statistic

```python
for value in stream:
    statistic.update(value)
```

#### Pattern C — Online supervised learning

```python
for x, y in stream:
    y_pred = model.predict_one(x)
    metric.update(y, y_pred)
    model.learn_one(x, y)
```

#### Pattern D — Monitor performance over time

```python
history.append(metric.get())
```

These patterns will appear repeatedly throughout the course.

# Gráfico com todas as observações e mudança da mean

In [ ]:
# Starter code for the optional mini-activity

my_values = [4, 5, 6, 5, 7, 6, 5, 8, 7, 6,
             5, 6, 7, 8, 7, 6, 5, 6, 7, 30]

running_means = []
mean = 0.0

for n, value in enumerate(my_values, start=1):
    mean += (value - mean) / n
    running_means.append(mean)

plt.figure(figsize=(9, 4))
plt.plot(my_values, marker="o", label="Observation")
plt.plot(running_means, label="Running mean")
plt.xlabel("Time")
plt.ylabel("Value")
plt.title("Streaming observations and incremental mean")
plt.legend()
plt.show()

# SAVE RAW AND PREPROCESSED DATA

In [ ]:
def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    return path


if ipma_obs_json:
    raw_ipma_path = save_json(
        ipma_obs_json,
        RAW_DIR / "ipma_observations_snapshot.json"
    )
    print("Saved raw data:", raw_ipma_path)

if not ipma_df.empty:
    processed_ipma_path = (
        PROCESSED_DIR / "ipma_observations_processed.csv"
    )
    ipma_df.to_csv(processed_ipma_path, index=False)
    print("Saved processed data:", processed_ipma_path)

# API DATA TO REAL STREAM

In [ ]:
def dataframe_replay_stream(
    df,
    timestamp_col,
    feature_cols,
    target_col=None
):
    """Yield timestamped observations one at a time."""

    ordered = df.sort_values(timestamp_col)

    for _, row in ordered.iterrows():
        timestamp = row[timestamp_col]

        x = {
            col: row[col]
            for col in feature_cols
            if col in row.index and pd.notna(row[col])
        }

        if target_col is None:
            yield timestamp, x
        else:
            y = row[target_col]
            yield timestamp, x, y

# SELECIONAR FEATURES
ipma_features = [
    "temperatura",
    "humidade",
    "pressao",
    "precAcumulada",
    "intensidadeVento"
]

if not selected_ipma.empty:
    stream = dataframe_replay_stream(
        selected_ipma,
        timestamp_col="timestamp",
        feature_cols=ipma_features
    )

    for i, observation in enumerate(stream):
        print(observation)

        if i >= 4:
            break
else:
    print("Load IPMA data first.")

# INCREMENTAL STATISTICS